# LC 72 — Edit Distance
**Day 65 | 2D Dynamic Programming | Hard**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> dp[i][j] = minimum edits to turn
the first i characters of word1 into the first j characters of
word2. If characters match — free. If not — pay 1 for the best
of insert, delete, or replace. Every sub-problem builds on three
neighbors in the table.
</div>

## Official Problem Statement

Given two strings `word1` and `word2`, return the minimum number
of operations required to convert `word1` to `word2`.

You have the following three operations permitted on a word:
- **Insert** a character
- **Delete** a character
- **Replace** a character

**Constraints:**
- `0 <= word1.length, word2.length <= 500`
- `word1` and `word2` consist of lowercase English letters.

## What This Is Actually Asking

You want to transform one string into another using the fewest
edits (insert, delete, replace). This is the classic Levenshtein
distance problem.

Think of it as aligning two strings character by character and
deciding at each position what the cheapest action is. The
decision at each step depends on previously computed sub-problems,
making this a natural fit for 2D DP.

The table has `(len(word1)+1)` rows and `(len(word2)+1)` columns.
Row 0 / column 0 represent empty string prefixes — their edit
distances are just the index values (insert/delete all chars).

## Walk Through an Example by Hand

**Input:** `word1="horse"`, `word2="ros"`

Initialize edges:
- `dp[i][0] = i` (delete i chars from word1)
- `dp[0][j] = j` (insert j chars to reach word2)

Fill dp[1][1]: `word1[0]='h'`, `word2[0]='r'` — not equal.
```
  min(dp[0][1]+1,  # delete from word1 -> 2
      dp[1][0]+1,  # insert into word1 -> 2
      dp[0][0]+1)  # replace           -> 1
  = 1
```

Fill dp[2][2]: `word1[1]='o'`, `word2[1]='o'` — equal!
```
  dp[2][2] = dp[1][1] = 1  (no cost, chars match)
```

Final answer: `dp[5][3] = 3`
(horse -> rorse -> rose -> ros)

## The Picture

```
  word1="horse"  word2="ros"

      "" r  o  s
  "" [ 0  1  2  3 ]
  h  [ 1  1  2  3 ]
  o  [ 2  2  1  2 ]
  r  [ 3  2  2  2 ]
  s  [ 4  3  3  2 ]
  e  [ 5  4  4  3 ]  <-- answer = 3

  Rule (chars differ):
    dp[i][j] = 1 + min(
      dp[i-1][j],    # delete from word1
      dp[i][j-1],    # insert into word1
      dp[i-1][j-1]   # replace
    )
  Rule (chars match):
    dp[i][j] = dp[i-1][j-1]  (free diagonal)
```

## When To Use This Pattern

- When asked for minimum cost to transform one sequence into
  another — think Edit Distance / Levenshtein DP.
- When you have two strings and must compare prefixes of each
  — think 2D DP table indexed by both string lengths.
- When the recurrence has three source cells (diagonal, above,
  left) — think insert/delete/replace triad.
- When characters match at position i,j — think free diagonal
  move (no cost, inherit dp[i-1][j-1]).
- When asked about spell-check, DNA alignment, or diff tools
  — think Edit Distance as the underlying model.

## The Approach

Build a 2D table of size `(m+1) x (n+1)` where m, n are string
lengths. Initialize the first row to `0..n` and first column to
`0..m`. For each cell (i, j), if `word1[i-1] == word2[j-1]`
carry the diagonal value unchanged. Otherwise take 1 plus the
minimum of the three neighbors: left (insert), above (delete),
diagonal (replace). The final answer is in `dp[m][n]`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run tests for min_distance (edit distance)."""
    tests = [
        # (word1, word2, expected)
        ("horse", "ros", 3),
        ("intention", "execution", 5),
        ("", "", 0),        # edge: both empty
        ("", "abc", 3),     # edge: word1 empty
        ("abc", "", 3),     # edge: word2 empty
        ("abc", "abc", 0),  # edge: identical
        ("a", "b", 1),
        ("kitten", "sitting", 3),
    ]
    passed = 0
    for word1, word2, expected in tests:
        result = func(word1, word2)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: '{word1}'->'{word2}' "
                f"expected={expected} got={result}"
            )
    total = len(tests)
    print(f"\nResults: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def min_distance(word1: str, word2: str) -> int:
    """
    Return the minimum edit distance (Levenshtein) between
    word1 and word2 using insert, delete, or replace.

    Approach: 2D DP table.
    - dp[i][j] = min edits to convert word1[:i] -> word2[:j].
    - Base: dp[i][0]=i, dp[0][j]=j (delete/insert all chars).
    - Transition:
        if word1[i-1] == word2[j-1]:
            dp[i][j] = dp[i-1][j-1]        # free match
        else:
            dp[i][j] = 1 + min(
                dp[i-1][j],   # delete from word1
                dp[i][j-1],   # insert into word1
                dp[i-1][j-1]  # replace
            )

    Args:
        word1: source string
        word2: target string

    Returns:
        Minimum number of operations (int)

    Examples:
        >>> min_distance("horse", "ros")
        3
        >>> min_distance("", "abc")
        3
    """
    # Debug: show input sizes
    print(
        f"[DEBUG] word1='{word1}' ({len(word1)}), "
        f"word2='{word2}' ({len(word2)})"
    )
    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(min_distance)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute Force (recursion) | O(3^(m+n)) | O(m+n) | Exponential branching |
| Memoized recursion | O(m*n) | O(m*n) | Top-down with cache |
| 2D DP table | O(m*n) | O(m*n) | Bottom-up, easy to trace |
| 1D rolling array | O(m*n) | O(n) | Keep only prev+curr row |

## Real World Connection

At **Citi**, reconciliation systems compare trade records from
different counterparties — edit distance measures how "far apart"
two transaction descriptions are to flag potential matches vs.
discrepancies. On **AWS**, services like Amazon Comprehend use
edit-distance-based algorithms for fuzzy entity matching in
unstructured documents. In **Data Engineering**, ETL pipelines
use Levenshtein distance to deduplicate customer records across
data sources with slightly different spellings or formats. Git's
diff algorithm is a close relative, computing the minimum number
of line-level edits between two file versions.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra